In [ ]:
from dotenv import load_dotenv
from hdbcli import dbapi

# Load environment variables
load_dotenv()

HANA_HOST = os.getenv("HANA_HOST")
HANA_PORT = os.getenv("HANA_PORT")
HANA_USER = os.getenv("HANA_USER")
HANA_PASSWORD = os.getenv("HANA_PASSWORD")

os.environ['AICORE_CLIENT_ID'] = os.getenv("AICORE_CLIENT_ID")
os.environ['AICORE_CLIENT_SECRET'] = os.getenv("AICORE_CLIENT_SECRET")
os.environ['AICORE_AUTH_URL'] = os.getenv("AICORE_AUTH_URL")
os.environ['AICORE_BASE_URL'] = os.getenv("AICORE_BASE_URL")
os.environ['AICORE_RESOURCE_GROUP'] = os.getenv("AICORE_RESOURCE_GROUP")

from gen_ai_hub.proxy.native.openai import chat

In [6]:
def query_database_with_sql_tool(user_query: str):
    """
    Generate SQL from natural language query and execute it against the HANA database.
    
    Args:
        user_query: Natural language description of the query
        
    Returns:
        Dictionary containing SQL, results, and metadata
    """
    try:
        schema_context = """
            You are an expert SQL developer working with a SAP HANA database schema named AI_HACKATHON.

            The database contains 1 table with the following schema:

            1. Table: AI_HACKATHON.WAREHOUSE (Master Data)
            - Artikelbezeichnung (NVARCHAR(500), PRIMARY KEY): Material description
            - Artikelnummer (NVARCHAR(50)): Description
            - Monatliche_Prognose_APO_ (DOUBLE): Monthly forecast from APO

            IMPORTANT NOTES:
            - To get actual inventory quantities and storage locations, use the get_material_inventory tool instead.
            - Use double quotes for schema/table/column names.
            - If needed, answer questions in German as well.
            - Use LIMIT to restrict results.
            """
        
        messages = [
            {"role": "system", "content": [{"type": "text", "text": schema_context}]},
            {"role": "user", "content": [{"type": "text", "text": f"Generate SQL for: {user_query}\nReturn ONLY the SQL statement. If this requires inventory data (quantities, storage locations, etc.), inform the user to use the inventory tool instead."}]}
        ]
        
        response = chat.completions.create(model_name="gpt-35-turbo", messages=messages)
        sql_statement = response.to_dict()["choices"][0]["message"]["content"].strip()
        
        # Clean SQL
        if sql_statement.startswith("```sql"):
            sql_statement = sql_statement[6:]
        if sql_statement.startswith("```"):
            sql_statement = sql_statement[3:]
        if sql_statement.endswith("```"):
            sql_statement = sql_statement[:-3]
        sql_statement = sql_statement.strip()
        
        conn = dbapi.connect(
            address=HANA_HOST,
            port=HANA_PORT,
            user=HANA_USER,
            password=HANA_PASSWORD
        )
        cursor = conn.cursor()
        cursor.execute(sql_statement)
        
        results = cursor.fetchall()
        column_names = [desc[0] for desc in cursor.description]
        
        cursor.close()
        conn.close()
        
        # Format results as list of dictionaries
        formatted_results = []
        for row in results[:50]:
            formatted_results.append(dict(zip(column_names, row)))
        
        return {
            "query": user_query,
            "sql_generated": sql_statement,
            "row_count": len(results),
            "results": formatted_results,
            "note": "Results limited to 50 rows" if len(results) > 50 else "All results shown"
        }
        
    except Exception as e:
        return {"error": f"Error executing database query: {str(e)}"}

In [7]:
query_database_with_sql_tool("Wie hoch ist die monatliche Bedarfsprognose des Artikels Kugellager?")

{'query': 'Wie hoch ist die monatliche Bedarfsprognose des Artikels Kugellager?',
 'sql_generated': 'SELECT "Monatliche_Prognose_APO_" \nFROM "AI_HACKATHON"."WAREHOUSE" \nWHERE "Artikelbezeichnung" = \'Kugellager\' \nLIMIT 1;',
 'row_count': 1,
 'results': [{'Monatliche_Prognose_APO_': 705.0}],
 'note': 'All results shown'}

In [8]:
def final_prompt_warehouse(orig_result):
    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": f"You are a warehouse management specialist focusing the quarterly management of warehouse items. You have previously asked the following question: '{orig_result['query']}'" +
                    f" The results from the database query are as follows: '{orig_result['results']}'. Based on these results, provide a concise and informative answer to the user's original question." +
                    "If asked in German, answer is German."
                }
            ]
        }
    ]
    return messages

In [ ]:
result = query_database_with_sql_tool("Wie hoch ist die monatliche Bedarfsprognose des Artikels Kugellager?")
final_messages = final_prompt_warehouse(result)
response = chat.completions.create(model_name="gpt-4o", messages=final_messages)
response_text = response.to_dict()["choices"][0]["message"]["content"].strip()
print(response_text)

Die monatliche Bedarfsprognose des Artikels Kugellager beträgt 705 Stück.
